In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['DATASET_25'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
os.environ['DATASET_15'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

print(os.path.exists(os.environ['DATASET_25']))
print(os.path.exists(os.environ['DATASET_15']))

sys.path.append('/content')

Mounted at /content/drive
True
True


In [ ]:
sys.path.append('/content/drive/MyDrive')

In [ ]:
# CATATAN (revisi 20-seed): notebook ini mengambil SEEDS dari common.py
# via `from common import ... SEEDS`. Pastikan common.py yang aktif di
# /content sudah versi 20-seed -- jalankan t2_revisi_fixed.ipynb (Step 0/1)
# atau t3_lstm_sensitivity_fixed.ipynb (Step 2) TERLEBIH DAHULU di runtime
# yang sama, karena kedua notebook itulah yang menulis ulang common.py.

"""
t1_leakage_gapsplit.py  —  T1 (review item C1)

Two parts:

  (A) Leakage audit table. For each regime: number of pseudo-episodes per
      class, and the distribution of temporal gaps (in 60-s bucket units)
      between train and test episodes of the SAME scenario. A gap of 1 means a
      train episode is temporally adjacent to a test episode -- the residual
      leakage the reviewer is worried about. Reported per seed and averaged.

  (B) Gap-split protocol. After the standard stratified split, remove from the
      TRAIN set any episode whose time-bucket is within `gap` buckets of a TEST
      episode in the same (rounded-speed, scenario) stream. Re-evaluate the two
      handcrafted configurations under this stricter split and compare against
      the standard split. If accuracy holds, the separability is not adjacency-
      driven; if it drops, residual temporal leakage is implicated.

The pseudo_run_id format is "<speed>_<scenario>_<bucket>", so the bucket index
and scenario are recovered by splitting the id string (no extra data needed).

NEEDS THE DATASET.
"""
import os, json
import numpy as np
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from common import (load_dataset, split_by_episode, create_windows,
                    pad_and_normalize, extract_tabular_features,
                    compute_metrics, agg, set_all_seeds, SEEDS, WINDOW_SEC,
                    CLASS_NAMES)


def parse_episode_id(ep):
    """'24.87_1_7' -> (scenario_int, bucket_int).
    Stream is defined by scenario ONLY, not by exact speed string — because
    speed fluctuates continuously within the ±2 m/s band, making exact-string
    matching fail to detect adjacency across episodes with slightly different
    recorded speeds."""
    parts = ep.rsplit('_', 2)   # ['24.87', '1', '7']
    return int(parts[1]), int(parts[2])


# ───────────────────────── (A) audit ───────────────────────────────────────
def audit_gaps(train_df, test_df):
    """Min temporal gap (in buckets) between each test episode and the nearest
    train episode of the same scenario stream. gap==1 => adjacent."""
    def buckets_by_stream(df):
        d = defaultdict(list)
        for ep in df['pseudo_run_id'].unique():
            sc, bk = parse_episode_id(ep)
            d[sc].append(bk)
        return d
    tr = buckets_by_stream(train_df)
    gaps = []
    for ep in test_df['pseudo_run_id'].unique():
        sc, bk = parse_episode_id(ep)
        cand = tr.get(sc, [])
        if cand:
            gaps.append(min(abs(bk - b) for b in cand))
    return gaps


# ───────────────────────── (B) gap-split ───────────────────────────────────
def apply_gap_split(train_df, test_df, gap=1):
    """Drop train episodes within `gap` buckets of any test episode (same scenario)."""
    test_buckets = defaultdict(set)
    for ep in test_df['pseudo_run_id'].unique():
        sc, bk = parse_episode_id(ep)
        test_buckets[sc].add(bk)
    keep_eps = []
    for ep in train_df['pseudo_run_id'].unique():
        sc, bk = parse_episode_id(ep)
        adj = any(abs(bk - tb) <= gap for tb in test_buckets.get(sc, set()))
        if not adj:
            keep_eps.append(ep)
    return train_df[train_df['pseudo_run_id'].isin(keep_eps)].copy()


def eval_handcrafted(train_df, val_df, test_df, seed):
    Xtr_l, ytr, _ = create_windows(train_df, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(val_df,   WINDOW_SEC)
    Xte_l, yte, _ = create_windows(test_df,  WINDOW_SEC)
    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        return None
    Xtr_s, Xv_s, Xte_s, _ = pad_and_normalize(Xtr_l, Xv_l, Xte_l)
    Xtr = extract_tabular_features(Xtr_s)
    Xv  = extract_tabular_features(Xv_s)
    Xte = extract_tabular_features(Xte_s)
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr); Xv = sc.transform(Xv); Xte = sc.transform(Xte)

    rf = RandomForestClassifier(n_estimators=200, max_depth=15,
                                min_samples_split=5, min_samples_leaf=2,
                                n_jobs=-1, random_state=seed,
                                class_weight='balanced')
    rf.fit(Xtr, ytr)
    rf_acc = compute_metrics(yte, rf.predict(Xte))['accuracy']

    xgb = XGBClassifier(objective='multi:softprob', num_class=3,
                        n_estimators=200, max_depth=5, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, random_state=seed,
                        eval_metric='mlogloss', early_stopping_rounds=20)
    xgb.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False)
    xgb_acc = compute_metrics(yte, xgb.predict(Xte))['accuracy']
    return rf_acc, xgb_acc, len(ytr)


def main(outdir='outputs_leakage', gap=1):
    os.makedirs(outdir, exist_ok=True)
    report = {}
    for spd in (25, 15):
        df = load_dataset(spd)
        per_seed = []
        for seed in SEEDS:
            set_all_seeds(seed)
            tr, va, te = split_by_episode(df, seed=seed)

            gaps = audit_gaps(tr, te)
            n_adjacent = int(sum(1 for g in gaps if g <= gap))

            std = eval_handcrafted(tr, va, te, seed)            # standard split
            tr_gap = apply_gap_split(tr, te, gap=gap)
            gp = eval_handcrafted(tr_gap, va, te, seed) if len(tr_gap) else None

            per_seed.append({
                'seed': seed,
                'n_episodes_total': int(df['pseudo_run_id'].nunique()),
                'min_gap_buckets': (int(min(gaps)) if gaps else None),
                'test_eps_with_adjacent_train': n_adjacent,
                'test_eps_total': len(gaps),
                'std_rf': std[0] if std else None,
                'std_xgb': std[1] if std else None,
                'gap_rf': gp[0] if gp else None,
                'gap_xgb': gp[1] if gp else None,
                'train_windows_std': std[2] if std else None,
                'train_windows_gap': gp[2] if gp else None,
            })
        report[f'{spd}ms'] = per_seed

    # console summary
    for spd in (25, 15):
        ps = report[f'{spd}ms']
        std_rf = agg([p['std_rf'] for p in ps if p['std_rf'] is not None])
        gap_rf = agg([p['gap_rf'] for p in ps if p['gap_rf'] is not None])
        std_xg = agg([p['std_xgb'] for p in ps if p['std_xgb'] is not None])
        gap_xg = agg([p['gap_xgb'] for p in ps if p['gap_xgb'] is not None])
        adj = np.mean([p['test_eps_with_adjacent_train'] for p in ps])
        tot = np.mean([p['test_eps_total'] for p in ps])
        print('\n' + '=' * 78)
        print(f'  T1 — Leakage audit & gap-split @ {spd} m/s (gap={gap} bucket)')
        print('=' * 78)
        print(f'  Test episodes adjacent to a train episode: {adj:.1f} / {tot:.1f} (avg/seed)')
        print(f'  RF-Stat      standard {std_rf[0]:6.2f}±{std_rf[1]:.2f}  ->  '
              f'gap-split {gap_rf[0]:6.2f}±{gap_rf[1]:.2f}  '
              f'(Δ {gap_rf[0]-std_rf[0]:+.2f} pp)')
        print(f'  XGBoost-only standard {std_xg[0]:6.2f}±{std_xg[1]:.2f}  ->  '
              f'gap-split {gap_xg[0]:6.2f}±{gap_xg[1]:.2f}  '
              f'(Δ {gap_xg[0]-std_xg[0]:+.2f} pp)')

    with open(os.path.join(outdir, 't1_leakage_gapsplit.json'), 'w') as f:
        json.dump(report, f, indent=2)
    print(f'\nSaved to {outdir}/')
    print('Reading: a large negative Δ under gap-split = adjacency-driven '
          'leakage; ~0 Δ = separability is not adjacency-driven.')


if __name__ == '__main__':
    main()


  T1 — Leakage audit & gap-split @ 25 m/s (gap=1 bucket)
  Test episodes adjacent to a train episode: 31.0 / 31.0 (avg/seed)
  RF-Stat      standard  97.65±5.26  ->  gap-split    nan±0.00  (Δ +nan pp)
  XGBoost-only standard  97.29±3.77  ->  gap-split    nan±0.00  (Δ +nan pp)

  T1 — Leakage audit & gap-split @ 15 m/s (gap=1 bucket)
  Test episodes adjacent to a train episode: 34.0 / 34.0 (avg/seed)
  RF-Stat      standard  72.28±9.01  ->  gap-split    nan±0.00  (Δ +nan pp)
  XGBoost-only standard  72.77±13.19  ->  gap-split    nan±0.00  (Δ +nan pp)

Saved to outputs_leakage/
Reading: a large negative Δ under gap-split = adjacency-driven leakage; ~0 Δ = separability is not adjacency-driven.


/content/drive/MyDrive/common.py:221: RuntimeWarning: Mean of empty slice.
  m, s = a.mean(), (a.std(ddof=1) if len(a) > 1 else 0.0)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
